**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Reinforcement Learning

Learning from *consequences* instead of labels — the paradigm the [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) name-dropped as RLHF and this course delivers. Five sessions: bandits, MDPs & Bellman, temporal-difference learning, policy gradients, and the road to RLHF — every algorithm verified against an exactly-solvable environment.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb) (expectations, LLN).
- [Training Dynamics](./Training_Dynamics.ipynb) for Session 4.
- Kinship worth knowing: TD learning's *new = old + α·(surprise)* is the [adaptive-filter heartbeat](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) yet again.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Bandits: Exploration vs Exploitation* (~35 min)
**Goal:** the RL problem with no states: regret, ε-greedy, and UCB's optimism.
**Feeds into:** Session 2 (MDPs).

---

## 2. The Ten-Armed Testbed

💡 **Intuition.** Ten slot machines, unknown payouts, 1000 pulls: every pull spent *learning* is a pull not spent *earning*. That tension — exploration vs exploitation — is RL's signature dilemma, isolated from everything else. **ε-greedy** explores blindly and forever; **UCB** explores *strategically*: pull the arm whose plausible upside $\hat\mu_a + c\sqrt{\ln t / n_a}$ is highest — 'optimism in the face of uncertainty', with the bonus shrinking exactly like a [confidence interval](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 2 of 5 — *MDPs & the Bellman Equation* (~40 min)
**Goal:** add states and time; solve a gridworld EXACTLY by dynamic programming.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (TD learning).

---

## 3. Markov Decision Processes

💡 **Intuition.** Now actions have *consequences that persist*: an MDP is states, actions, transition probabilities, rewards, and a discount $\gamma$. The value $V^\pi(s)$ is expected discounted return — and Bellman's equation says value is **recursively self-consistent**: today's value = today's reward + γ·tomorrow's value. The optimal version ($V^* = \max_a [r + \gamma E V^*]$) is a fixed-point equation, and **value iteration** just applies it until it stops moving — a contraction ([Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb): Cauchy convergence with rate γ!). This gives us an *exact oracle* to test every learning algorithm against.

In [ ]:
# 4x4 gridworld: start anywhere, goal at (3,3) reward +1, pit at (1,2) reward −1, step −0.02
# value iteration = the exact solution (our ORACLE for everything later)

# YOUR CODE HERE


---
### 🕐 Session 3 of 5 — *Temporal-Difference Learning* (~40 min)
**Goal:** learn the same values WITHOUT the model: TD(0) and Q-learning, checked against the oracle.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (policy gradients).

---

## 4. Learning from the Surprise

💡 **Intuition.** Value iteration needed the transition model. An *agent* only gets experience: $(s, a, r, s')$. TD's move: use the Bellman equation as an **error signal** — the *TD error* $\delta = r + \gamma \max_a Q(s', a) - Q(s, a)$ is how surprised you are, and $Q \mathrel{+}= \alpha \delta$ is the [LMS update](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the 'desired signal'. Q-learning does this off-policy (learns the greedy value while exploring) — and, on a finite MDP with decaying exploration, provably converges to $Q^*$. We *check* that, since we own the oracle.

In [ ]:
# tie-aware policy check: the learned greedy action must be (near-)optimal under Q*

# YOUR CODE HERE


---
### 🕐 Session 4 of 5 — *Policy Gradients* (~40 min)
**Goal:** skip values, optimize the policy directly: REINFORCE with a baseline, from scratch.
**Builds on:** Session 3; [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 5 (the road to RLHF).

---

## 5. Differentiating Through Luck

💡 **Intuition.** Values are a detour; why not adjust the policy's parameters to make good episodes more likely? The log-derivative trick makes the un-differentiable differentiable: $\nabla E[R] = E[R \, \nabla \log \pi(a|s)]$ — *reinforce the log-probability of what you did, in proportion to how well it went*. The estimator is unbiased but wildly noisy ([SGD's](../Intro_Math/Optimization/Optimization.ipynb) noise-floor problem, squared); subtracting a **baseline** (the mean return) cancels variance without adding bias — the single most important practical trick in policy-land.

In [ ]:
# REINFORCE on the same gridworld (tabular softmax policy)

# YOUR CODE HERE


---
### 🕐 Session 5 of 5 — *The Road to RLHF* (~30 min)
**Goal:** connect this course to modern practice: reward models, KL anchors, and PPO's role.
**Builds on:** Session 4.

---

## 6. From Gridworld to Chatbots

The [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) said 'preference tuning shapes judgment'; you now have the vocabulary for how:

1. **The policy** is the language model; a *state* is the prompt + text so far, an *action* is the next token, an *episode* is a completion.
2. **The reward** comes from a *reward model* trained on human preference pairs — [cross-entropy](../Intro_Math/Information_Theory/Information_Theory.ipynb) on 'which answer did the human prefer'.
3. **The optimizer** is a policy gradient with variance-reduction armor: PPO ≈ REINFORCE + a learned baseline (critic) + a *trust region* (clipped updates — don't move the policy further than the reward model's validity extends).
4. **The KL anchor**: reward is penalized by KL divergence from the pretrained model — 'improve preferences *without leaving the language manifold*'. DPO folds reward model + RL into one supervised loss on preference pairs, which is why it took over.

💡 **Intuition.** Everything hard about RLHF is Session 4's variance problem wearing a $10^{11}$-parameter costume, plus one new failure mode this course equips you to name: **reward hacking** — the policy exploiting the reward model where it's [off-distribution](./Uncertainty_in_ML.ipynb).

## 7. Conclusion

Bandits isolate exploration; Bellman makes value self-consistent; TD learns it from surprise (LMS's heartbeat again); policy gradients differentiate through luck with a baseline as armor; RLHF is all four at industrial scale. Every algorithm here was checked against an exact oracle — a habit worth keeping when the environments stop being 4×4.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — the policy being tuned.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — the SGD theory under the noise.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-state tracking: what 'state' means when you can't see it.